# Atributos de instancia vs atributos de clase

Resumen:

- **`self.algo = ...`** → atributo de **instancia** (cada objeto tiene el suyo).
- **`algo = ...` en el cuerpo de `class`** → atributo de **clase** (compartido por todas las instancias).
- **`self.algo = ...` dentro de un método** → sigue siendo atributo de **instancia**, aunque no esté en `__init__`.
- **`Clase.algo = ...` dentro de un método** → modifica el atributo de **clase** (compartido).

## 1. Atributo de instancia (`self.a`)

Cada instancia guarda su propio valor. Cambiar `obj1.a` no afecta a `obj2.a`.

In [ ]:
class Usuario:
    def __init__(self, nombre):
        self.nombre = nombre  # atributo de instancia

a = Usuario("Ana")
b = Usuario("Luis")

a.nombre = "María"
print("a.nombre:", a.nombre)
print("b.nombre:", b.nombre)  # Luis — no cambió

## 2. Atributo de clase (compartido)

Definido en la clase, sin `self`. Todas las instancias leen el mismo valor (hasta que una instancia "sombrea" con `self.x = ...`).

In [ ]:
class Equipo:
    limite_miembros = 5  # atributo de clase

    def __init__(self, nombre):
        self.nombre = nombre  # atributo de instancia

e1 = Equipo("Alpha")
e2 = Equipo("Beta")

print("e1.limite_miembros:", e1.limite_miembros)
print("e2.limite_miembros:", e2.limite_miembros)
print("¿Mismo objeto?", e1.limite_miembros is e2.limite_miembros)

Equipo.limite_miembros = 10  # cambio en la clase
print("Tras cambiar la clase → e1:", e1.limite_miembros, "e2:", e2.limite_miembros)

---

## 3. Argumento del método vs guardar en `self` (estado por usuario)

Imagina dos sesiones de chat (`sesion_ana`, `sesion_luis`). Cada una tiene un `user_id` fijado en `__init__`.
El método `registrar_preferencia(preferencia_nueva)` recibe un texto **solo para esa llamada**.

- **`preferencia_nueva`**: parámetro **local**; desaparece al terminar el método si no lo copias a otro sitio.
- **`self.ultima_preferencia = preferencia_nueva`**: conviertes ese dato en **atributo de instancia** del usuario que llamó (`self`).

No hay una `ultima_preferencia` global de la clase: **`sesion_ana.ultima_preferencia` y `sesion_luis.ultima_preferencia` son independientes**.

In [ ]:
class SesionChat:
    """Cada instancia = un usuario distinto en el sistema multiusuario."""

    def __init__(self, user_id: str):
        self.user_id = user_id  # atributo de instancia (fijado al crear la sesión)

    def registrar_preferencia(self, preferencia_nueva: str) -> str:
        # preferencia_nueva solo existe mientras corre esta llamada...
        self.ultima_preferencia = preferencia_nueva  # ...pero aquí la guardamos EN ESTE usuario
        return preferencia_nueva

sesion_ana = SesionChat("ana")
sesion_luis = SesionChat("luis")

print("Ana registra →", sesion_ana.registrar_preferencia("café con leche"))
print("Luis registra →", sesion_luis.registrar_preferencia("té verde"))

print("Estado Ana:", sesion_ana.user_id, "→", sesion_ana.ultima_preferencia)
print("Estado Luis:", sesion_luis.user_id, "→", sesion_luis.ultima_preferencia)

sesion_ana.registrar_preferencia("descafeinado")  # solo cambia el estado de Ana
print("Tras actualizar Ana:", sesion_ana.ultima_preferencia)
print("Luis no cambió:", sesion_luis.ultima_preferencia)

## 3.1 Variante: `ultima_preferencia = preferencia_nueva` (sin `self`)

Si escribes **`ultima_preferencia = preferencia_nueva`** (sin `self.`), creas una **variable local del método**, no un atributo del objeto.

- El `return preferencia_nueva` **sí funciona**: devuelves el texto de esa llamada.
- **`sesion_ana.ultima_preferencia` no existe** después: no guardaste nada en la instancia.
- Al terminar el método, `ultima_preferencia` local **se destruye**; la próxima llamada empieza de cero.

Comparación rápida:

| Asignación | ¿Queda en el objeto? | ¿Otro método puede leerlo después? |
|------------|----------------------|-------------------------------------|
| `self.ultima_preferencia = ...` | Sí (atributo de instancia) | Sí, con `self.ultima_preferencia` |
| `ultima_preferencia = ...` | No (solo local al método) | No |

In [ ]:
class SesionChatSinGuardar:
    def __init__(self, user_id: str):
        self.user_id = user_id

    def registrar_preferencia(self, preferencia_nueva: str) -> str:
        ultima_preferencia = preferencia_nueva  # local: no es self.ultima_preferencia
        return ultima_preferencia  # el return sigue bien; lo local no queda en la sesión

sesion_ana = SesionChatSinGuardar("ana")
valor_devuelto = sesion_ana.registrar_preferencia("café con leche")
print("Valor devuelto:", valor_devuelto)
print("user_id sigue en la instancia:", sesion_ana.user_id)

if hasattr(sesion_ana, "ultima_preferencia"):
    print("ultima_preferencia guardada:", sesion_ana.ultima_preferencia)
else:
    print("La sesión NO tiene atributo ultima_preferencia (no usamos self.)")

# Segunda llamada: no hay "memoria" en el objeto de la preferencia anterior
sesion_ana.registrar_preferencia("té verde")
print("Tras otra llamada, ¿ultima_preferencia?", hasattr(sesion_ana, "ultima_preferencia"))

---

## Para recordar: `self.algo = ...` en `__init__` vs en un método

En Python **no hay una lista fija de atributos** como en otros lenguajes: el objeto los va ganando cuando haces **`self.nombre = valor`**.

**Mecánica (igual en ambos sitios):**  
Tanto en `__init__` como en `registrar_preferencia` (o en `_init_vector_db()`), **`self.algo = ...` crea o actualiza un atributo de instancia** en *ese* objeto (`sesion_ana`, un `ModernMemoryManager`, etc.). No es “enviar” el valor a la clase.

**Lo que sí cambia es el momento:**

| Dónde asignas | Cuándo existe | Uso habitual |
|---------------|---------------|--------------|
| `__init__` | Al crear el objeto (`Objeto(...)`) | Estado inicial que **siempre** necesitas (`user_id`, rutas, conexiones) |
| Otro método | Cuando se llama ese método (puede ser nunca) | Estado que **aparece o cambia** con el uso (última preferencia, último turno) |
| Método `_init_*` llamado **desde** `__init__` | También al crear el objeto | Organizar código (ej. `memory_manager`: `_init_vector_db()` → `self.vectorstore`) |

**Convención útil:** si otros métodos van a leer `self.ultima_preferencia`, puedes **reservarla en `__init__`** con `None` (o `[]`) y **actualizarla** después en un método — es el **mismo atributo**, solo que garantizas que “existe” desde el principio.

**Sin `self.`** (variable local o solo `return`): el dato **no queda en el objeto**, aunque el método devuelva un valor al caller.

In [ ]:
# Misma mecánica self.* ; distinto MOMENTO en que el atributo "tiene sentido"

class SesionInitExplicito:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.ultima_preferencia = None  # atributo de instancia desde el minuto 0

    def registrar(self, preferencia: str):
        self.ultima_preferencia = preferencia  # actualiza el mismo atributo

class SesionAtributoTardio:
    def __init__(self, user_id: str):
        self.user_id = user_id
        # self.ultima_preferencia aún no existe

    def registrar(self, preferencia: str):
        self.ultima_preferencia = preferencia  # aquí nace por primera vez

tardia = SesionAtributoTardio("luis")
print("Antes de registrar:", hasattr(tardia, "ultima_preferencia"))
tardia.registrar("té verde")
print("Después:", tardia.ultima_preferencia)

explicita = SesionInitExplicito("ana")
print("Desde init:", explicita.ultima_preferencia)  # None, pero el atributo ya existe
explicita.registrar("café")
print("Tras registrar:", explicita.ultima_preferencia)

## 4. Trampa: `self.contador += 1` con contador de clase

`self.contador += 1` **no** incrementa el atributo de clase: crea un contador **por instancia**.
Para modificar el compartido usa **`Clase.contador += 1`**.

In [ ]:
class ContadorMal:
    total = 0

    def incrementar_mal(self):
        self.total += 1  # sombrea: atributo de instancia

class ContadorBien:
    total = 0

    def incrementar_bien(self):
        ContadorBien.total += 1

x = ContadorMal()
y = ContadorMal()
x.incrementar_mal()
y.incrementar_mal()
print("ContadorMal.total (clase):", ContadorMal.total)  # sigue 0
print("x.total, y.total (instancia):", x.total, y.total)  # 1, 1

a = ContadorBien()
b = ContadorBien()
a.incrementar_bien()
b.incrementar_bien()
print("ContadorBien.total (clase):", ContadorBien.total)  # 2

## 5. Lista/dict de clase: mutar compartido vs reasignar

`.append()` sobre `self.items` **muta la lista compartida** (afecta a todas las instancias).
`self.items = []` crea una lista **solo en esa instancia**.

In [ ]:
class Caja:
    items = []  # ¡compartida entre instancias!

    def agregar(self, x):
        self.items.append(x)

c1 = Caja()
c2 = Caja()
c1.agregar("A")
c2.agregar("B")
print("c1.items:", c1.items)
print("c2.items:", c2.items)  # misma lista: ['A', 'B']
print("¿Misma lista?", c1.items is c2.items)

## 6. Analogía con `ModernMemoryManager`

En el chat multiusuario, `self.user_id`, `self.vectorstore`, `self.memory_parser` son **atributos de instancia**:
un manager por usuario, con su carpeta y su Chroma. No se comparten entre usuarios.

In [ ]:
# Patrón simplificado (sin LangChain/Chroma)
class MemoryManagerDemo:
    DEFAULT_MAX_RESULTS = 5  # atributo de clase: igual para todos

    def __init__(self, user_id: str):
        self.user_id = user_id
        self.memorias = []  # atributo de instancia: por usuario

u1 = MemoryManagerDemo("alice")
u2 = MemoryManagerDemo("bob")
u1.memorias.append("Me gusta el café")

print("u1:", u1.user_id, u1.memorias)
print("u2:", u2.user_id, u2.memorias)
print("MAX compartido:", u1.DEFAULT_MAX_RESULTS, u2.DEFAULT_MAX_RESULTS)